In [1]:
# ============================================================
# 15_US_proxy_factor_transfer_forecasting.ipynb
#
# AURORA-TWETF extension:
# U.S. proxy-factor transfer experiment
#
# Purpose:
# 1. Test whether lagged U.S. market proxy factors improve
#    Taiwan ETF regime forecasting.
# 2. Compare feature sets:
#       - TW-only
#       - TW + U.S. broad market
#       - TW + U.S. semiconductor
#       - TW + U.S. risk / macro
#       - TW + all U.S. proxy factors
# 3. Use leakage-controlled purged / embargoed walk-forward
#    evaluation for 20d and 60d ordinal regimes.
# 4. Produce paper-ready tables, figures, and manuscript text.
#
# Key leakage rule:
# Taiwan day t can only use U.S. market information known before
# the Taiwan decision. Therefore all U.S. proxy features are shifted
# by at least one trading day before rolling features are computed.
#
# Educational / research use only.
# Not personalized financial advice.
# ============================================================

from __future__ import annotations

import json
import math
import hashlib
import warnings
from pathlib import Path
from datetime import datetime, timezone

warnings.filterwarnings("ignore")

# ============================================================
# 0. Colab setup
# ============================================================

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Google Drive mount skipped or failed.")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    HAS_SEABORN = False

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    cohen_kappa_score,
    log_loss,
    confusion_matrix,
)
from sklearn.calibration import CalibratedClassifierCV

# Optional models.
HAS_LGBM = False
HAS_XGB = False

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except Exception:
    pass

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except Exception:
    pass

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ============================================================
# 1. Paths and run configuration
# ============================================================

PROJECT_ROOT = Path("/content/drive/MyDrive/AURORA_TWETF")
PROJECT_CODE = "AURORA_TWETF"
COMPARISON_CODE = "ROMA_AURORA_TWETF"

MODELING_DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "modeling"
    / "AURORA_TWETF_features_with_labels.parquet"
)

ETF_RETURN_PANEL_PATH = (
    PROJECT_ROOT
    / "data"
    / "panels"
    / "AURORA_etf_return_panel.parquet"
)

PANELS_DIR = PROJECT_ROOT / "data" / "panels"

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "outputs"
    / COMPARISON_CODE
    / "us_proxy_factor_transfer_forecasting"
)

GLOBAL_TABLE_DIR = PROJECT_ROOT / "outputs" / COMPARISON_CODE / "tables"
GLOBAL_FIGURE_DIR = PROJECT_ROOT / "outputs" / COMPARISON_CODE / "figures"
GLOBAL_REPORT_DIR = PROJECT_ROOT / "outputs" / COMPARISON_CODE / "reports"
GLOBAL_MANUSCRIPT_DIR = PROJECT_ROOT / "outputs" / COMPARISON_CODE / "manuscript_assets"

RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

RUN_ROOT = OUTPUT_ROOT / f"run_{RUN_ID}"

TABLE_RUN_DIR = RUN_ROOT / "tables"
DATA_RUN_DIR = RUN_ROOT / "data"
PRED_RUN_DIR = RUN_ROOT / "predictions"
PROBA_RUN_DIR = RUN_ROOT / "probabilities"
MODEL_RUN_DIR = RUN_ROOT / "models"
FIGURE_RUN_DIR = RUN_ROOT / "figures"
PAPER_FIGURE_DIR = RUN_ROOT / "paper_figures"
REPORT_RUN_DIR = RUN_ROOT / "reports"
MANUSCRIPT_RUN_DIR = RUN_ROOT / "manuscript_assets"

for d in [
    RUN_ROOT,
    TABLE_RUN_DIR,
    DATA_RUN_DIR,
    PRED_RUN_DIR,
    PROBA_RUN_DIR,
    MODEL_RUN_DIR,
    FIGURE_RUN_DIR,
    PAPER_FIGURE_DIR,
    REPORT_RUN_DIR,
    MANUSCRIPT_RUN_DIR,
    GLOBAL_TABLE_DIR,
    GLOBAL_FIGURE_DIR,
    GLOBAL_REPORT_DIR,
    GLOBAL_MANUSCRIPT_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("Notebook 15: U.S. Proxy-Factor Transfer Forecasting")
print("=" * 100)
print("Timestamp UTC        :", RUN_TIMESTAMP)
print("Run ID               :", RUN_ID)
print("Project root         :", PROJECT_ROOT)
print("Modeling data        :", MODELING_DATA_PATH)
print("ETF return panel     :", ETF_RETURN_PANEL_PATH)
print("Panels directory     :", PANELS_DIR)
print("Run root             :", RUN_ROOT)
print("=" * 100)

if not MODELING_DATA_PATH.exists():
    raise FileNotFoundError(f"Missing modeling data: {MODELING_DATA_PATH}")

# ============================================================
# 2. Global experiment settings
# ============================================================

ETF_TICKERS = ["0050", "006208", "00692", "00881"]
ORDINAL_CLASSES = np.array([0, 1, 2, 3, 4])
HORIZONS = [20, 60]

# Feature windows for lagged U.S. proxy features.
LAG_WINDOWS = [1, 2, 5, 10, 20]
ROLL_WINDOWS = [5, 10, 20, 60]

# Walk-forward settings follow the AURORA / ROMA purged design.
# If the dataset length is shorter than expected, the notebook
# automatically falls back to adaptive folds.
EXPECTED_N = 1262

TRANSACTION_NOTE = (
    "Notebook 15 evaluates forecasting only. Allocation and transaction-cost "
    "sensitivity remain covered by Notebook 14B."
)

# ============================================================
# 3. Utility functions
# ============================================================

def read_table(path: Path) -> pd.DataFrame:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    raise ValueError(f"Unsupported file type: {path}")

def save_table(df: pd.DataFrame, local_name: str, global_name: str | None = None):
    local_path = TABLE_RUN_DIR / local_name
    df.to_csv(local_path, index=False)
    global_path = None
    if global_name is not None:
        global_path = GLOBAL_TABLE_DIR / global_name
        df.to_csv(global_path, index=False)
    return local_path, global_path

def save_parquet_csv(df: pd.DataFrame, base_path: Path, index=True):
    csv_path = Path(str(base_path) + ".csv")
    parquet_path = Path(str(base_path) + ".parquet")
    df.to_csv(csv_path, index=index)
    try:
        df.to_parquet(parquet_path, index=index)
    except Exception as e:
        print("Parquet save skipped:", parquet_path, repr(e))
        parquet_path = None
    return parquet_path, csv_path

def write_json(path: Path, obj) -> None:
    path.write_text(
        json.dumps(obj, indent=2, ensure_ascii=False, default=str),
        encoding="utf-8",
    )

def write_markdown(path: Path, text: str) -> None:
    path.write_text(text, encoding="utf-8")

def sha256_file(path: Path, chunk_size=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def make_file_manifest(root: Path) -> pd.DataFrame:
    rows = []
    for p in sorted(Path(root).rglob("*")):
        if p.is_file():
            stat = p.stat()
            rows.append({
                "path": p.relative_to(root).as_posix(),
                "size_bytes": int(stat.st_size),
                "modified_utc": datetime.fromtimestamp(
                    stat.st_mtime,
                    timezone.utc,
                ).strftime("%Y-%m-%dT%H:%M:%SZ"),
                "sha256": sha256_file(p),
            })
    return pd.DataFrame(rows)

def standardize_date_index(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    if "date" in out.columns:
        out["date"] = pd.to_datetime(out["date"])
        out = out.set_index("date")
    else:
        out.index = pd.to_datetime(out.index)
    out = out.sort_index()
    out.index.name = "date"
    return out

def clean_colname(x: str) -> str:
    return (
        str(x)
        .strip()
        .replace(" ", "_")
        .replace("/", "_")
        .replace("-", "_")
        .replace(".", "_")
        .replace("^", "")
        .replace("=", "")
        .replace("(", "")
        .replace(")", "")
        .replace("%", "pct")
    )

def safe_numeric_df(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for c in out.columns:
        out[c] = pd.to_numeric(out[c], errors="coerce")
    return out

def ordinal_mae(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.mean(np.abs(y_true - y_pred)))

def expected_class_from_proba(proba: np.ndarray, classes=ORDINAL_CLASSES):
    proba = np.asarray(proba)
    return proba @ np.asarray(classes)

def ordinal_rmse_expected(y_true, proba, classes=ORDINAL_CLASSES):
    exp_y = expected_class_from_proba(proba, classes)
    return float(np.sqrt(np.mean((np.asarray(y_true) - exp_y) ** 2)))

def ece_multiclass(y_true, proba, n_bins=10):
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)
    pred = proba.argmax(axis=1)
    conf = proba.max(axis=1)
    correct = (pred == y_true).astype(float)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        if i == n_bins - 1:
            mask = (conf >= lo) & (conf <= hi)
        else:
            mask = (conf >= lo) & (conf < hi)
        if mask.sum() == 0:
            continue
        ece += (mask.mean()) * abs(correct[mask].mean() - conf[mask].mean())
    return float(ece)

def brier_multiclass(y_true, proba, classes=ORDINAL_CLASSES):
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)
    y_onehot = np.zeros_like(proba, dtype=float)
    class_to_idx = {c: i for i, c in enumerate(classes)}
    for i, y in enumerate(y_true):
        if y in class_to_idx:
            y_onehot[i, class_to_idx[y]] = 1.0
    return float(np.mean(np.sum((proba - y_onehot) ** 2, axis=1)))

def align_proba_to_classes(model_classes, proba, target_classes=ORDINAL_CLASSES):
    model_classes = np.asarray(model_classes)
    proba = np.asarray(proba)
    out = np.zeros((proba.shape[0], len(target_classes)), dtype=float)

    for j, cls in enumerate(model_classes):
        if cls in target_classes:
            idx = np.where(target_classes == cls)[0][0]
            out[:, idx] = proba[:, j]

    row_sum = out.sum(axis=1)
    missing = row_sum <= 1e-12
    out[~missing] = out[~missing] / row_sum[~missing, None]
    out[missing, :] = 1.0 / len(target_classes)
    return out

def evaluate_classification(y_true, y_pred, proba=None, classes=ORDINAL_CLASSES):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    metrics = {
        "n": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "ordinal_mae": ordinal_mae(y_true, y_pred),
        "quadratic_weighted_kappa": float(cohen_kappa_score(y_true, y_pred, weights="quadratic")),
    }

    if proba is not None:
        proba = np.asarray(proba)
        try:
            metrics["log_loss"] = float(log_loss(y_true, proba, labels=list(classes)))
        except Exception:
            metrics["log_loss"] = np.nan
        metrics["brier"] = brier_multiclass(y_true, proba, classes=classes)
        metrics["ece"] = ece_multiclass(y_true, proba)
        metrics["ordinal_rmse_expected"] = ordinal_rmse_expected(y_true, proba, classes=classes)
    else:
        metrics["log_loss"] = np.nan
        metrics["brier"] = np.nan
        metrics["ece"] = np.nan
        metrics["ordinal_rmse_expected"] = np.nan

    return metrics

def rank_composite(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    rank_specs = {
        "accuracy": False,
        "balanced_accuracy": False,
        "macro_f1": False,
        "quadratic_weighted_kappa": False,
        "ordinal_mae": True,
        "log_loss": True,
        "brier": True,
        "ece": True,
    }

    rank_cols = []
    for col, ascending in rank_specs.items():
        if col in out.columns:
            rcol = f"rank_{col}"
            out[rcol] = out[col].rank(ascending=ascending, method="min")
            rank_cols.append(rcol)

    if rank_cols:
        out["composite_rank"] = out[rank_cols].mean(axis=1)
    else:
        out["composite_rank"] = np.nan

    return out

def find_panel_files():
    rows = []
    for p in sorted(PANELS_DIR.rglob("*")):
        if p.is_file() and p.suffix.lower() in [".parquet", ".csv"]:
            try:
                df = read_table(p)
                rows.append({
                    "path": str(p),
                    "name": p.name,
                    "shape": str(df.shape),
                    "n_rows": int(df.shape[0]),
                    "n_cols": int(df.shape[1]),
                    "columns_preview": ", ".join(map(str, df.columns[:20])),
                })
            except Exception as e:
                rows.append({
                    "path": str(p),
                    "name": p.name,
                    "shape": "read_failed",
                    "n_rows": 0,
                    "n_cols": 0,
                    "columns_preview": repr(e),
                })
    return pd.DataFrame(rows)

def select_best_return_panel(panel_index: pd.DataFrame):
    if panel_index.empty:
        return None

    candidates = panel_index.copy()
    candidates["score"] = 0
    candidates.loc[candidates["name"].str.lower().str.contains("return"), "score"] += 5
    candidates.loc[candidates["name"].str.lower().str.contains("panel"), "score"] += 2
    candidates.loc[candidates["n_cols"] >= 8, "score"] += 2
    candidates.loc[candidates["n_rows"] >= 500, "score"] += 1

    candidates = candidates.sort_values(["score", "n_cols", "n_rows"], ascending=[False, False, False])
    if candidates.iloc[0]["score"] <= 0:
        return None
    return Path(candidates.iloc[0]["path"])

# ============================================================
# 4. Load modeling data and identify targets
# ============================================================

print("\n" + "=" * 100)
print("Step 1: Load modeling data and identify target columns")
print("=" * 100)

model_df = standardize_date_index(read_table(MODELING_DATA_PATH))
model_df.columns = [clean_colname(c) for c in model_df.columns]
model_df = model_df.sort_index()

print("Modeling dataset shape:", model_df.shape)
print("Date range:", model_df.index.min(), "to", model_df.index.max())
print("Columns preview:", model_df.columns[:30].tolist())

def infer_target_columns(df: pd.DataFrame):
    cols = list(df.columns)
    lower_map = {c: c.lower() for c in cols}

    target_cols = {}

    for h in HORIZONS:
        candidates = []
        for c in cols:
            lc = lower_map[c]
            if str(h) in lc and any(k in lc for k in ["target", "label", "regime", "class", "y_"]):
                s = pd.to_numeric(df[c], errors="coerce").dropna()
                if len(s) and set(s.astype(int).unique()).issubset(set(ORDINAL_CLASSES)):
                    candidates.append(c)

        priority = []
        for c in candidates:
            lc = lower_map[c]
            score = 0
            if f"{h}d" in lc:
                score += 4
            if f"h{h}" in lc:
                score += 3
            if "regime" in lc:
                score += 3
            if "target" in lc:
                score += 2
            if "label" in lc:
                score += 2
            if "return" in lc:
                score -= 2
            priority.append((score, c))

        if priority:
            priority = sorted(priority, reverse=True)
            target_cols[h] = priority[0][1]

    return target_cols

TARGET_COLS = infer_target_columns(model_df)

print("Inferred target columns:", TARGET_COLS)

if set(TARGET_COLS.keys()) != set(HORIZONS):
    print("\nCould not infer both target columns automatically.")
    print("Candidate ordinal columns:")
    candidate_rows = []
    for c in model_df.columns:
        s = pd.to_numeric(model_df[c], errors="coerce").dropna()
        if len(s) and len(s.unique()) <= 10:
            vals = sorted(s.astype(int).unique().tolist()) if np.all(np.isclose(s, s.astype(int))) else sorted(s.unique().tolist())[:10]
            candidate_rows.append({
                "column": c,
                "n_unique": len(s.unique()),
                "values": vals,
            })
    cand_df = pd.DataFrame(candidate_rows)
    print(cand_df.head(80).to_string(index=False))

    raise ValueError(
        "Please manually set TARGET_COLS, for example: "
        "TARGET_COLS = {20: 'target_20d', 60: 'target_60d'}"
    )

target_audit_rows = []
for h, c in TARGET_COLS.items():
    dist = model_df[c].value_counts(dropna=False).sort_index()
    for cls, count in dist.items():
        target_audit_rows.append({
            "horizon": h,
            "target_column": c,
            "class": cls,
            "count": int(count),
            "share": float(count / len(model_df)),
        })

target_audit = pd.DataFrame(target_audit_rows)

save_table(
    target_audit,
    "notebook15_00_target_distribution_audit.csv",
    f"table_15_00_target_distribution_audit_{RUN_ID}.csv",
)

# ============================================================
# 5. Load raw return panels and construct lagged U.S. proxy features
# ============================================================

print("\n" + "=" * 100)
print("Step 2: Load raw panels and construct lagged U.S. proxy features")
print("=" * 100)

panel_index = find_panel_files()

save_table(
    panel_index,
    "notebook15_01_available_panel_files.csv",
    f"table_15_01_available_panel_files_{RUN_ID}.csv",
)

print("Panel files:")
if len(panel_index):
    print(panel_index.to_string(index=False))
else:
    print("No panel files found.")

raw_return_path = select_best_return_panel(panel_index)

if raw_return_path is None:
    print("No broad return panel found. Notebook will rely on existing modeling features only.")
    raw_ret = pd.DataFrame(index=model_df.index)
else:
    raw_ret = standardize_date_index(read_table(raw_return_path))
    raw_ret.columns = [clean_colname(c) for c in raw_ret.columns]
    raw_ret = safe_numeric_df(raw_ret)
    print("Selected raw return panel:", raw_return_path)
    print("Raw return panel shape:", raw_ret.shape)
    print("Raw return panel columns:", raw_ret.columns.tolist())

raw_ret = raw_ret.reindex(model_df.index)

def classify_proxy_columns(columns):
    broad = []
    semi = []
    risk = []
    fx = []
    asia = []
    tw_local = []

    for c in columns:
        lc = c.lower()

        if any(k in lc for k in ["0050", "006208", "00692", "00881", "taiex", "taiwan", "twii", "twse"]):
            tw_local.append(c)

        if any(k in lc for k in ["sp500", "s_p500", "s&p", "spx", "spy", "nasdaq", "ixic", "ndx", "qqq"]):
            broad.append(c)

        if any(k in lc for k in ["soxx", "smh", "semiconductor", "semi", "philadelphia_semiconductor", "sox"]):
            semi.append(c)

        if any(k in lc for k in ["vix", "us10", "10y", "yield", "treasury", "rate"]):
            risk.append(c)

        if any(k in lc for k in ["usd", "twd", "usdtwd", "dxy", "fx"]):
            fx.append(c)

        if any(k in lc for k in ["nikkei", "n225", "hang", "hsi", "kospi", "korea", "japan", "hong"]):
            asia.append(c)

    return {
        "tw_local": sorted(set(tw_local)),
        "us_broad": sorted(set(broad)),
        "us_semi": sorted(set(semi)),
        "us_risk": sorted(set(risk)),
        "fx": sorted(set(fx)),
        "asia": sorted(set(asia)),
    }

proxy_groups_raw = classify_proxy_columns(raw_ret.columns)

proxy_group_rows = []
for group, cols in proxy_groups_raw.items():
    for c in cols:
        proxy_group_rows.append({
            "raw_group": group,
            "raw_column": c,
        })

proxy_group_audit = pd.DataFrame(proxy_group_rows)

save_table(
    proxy_group_audit,
    "notebook15_02_raw_proxy_group_audit.csv",
    f"table_15_02_raw_proxy_group_audit_{RUN_ID}.csv",
)

print("Raw proxy groups:")
for g, cols in proxy_groups_raw.items():
    print(f"{g}: {cols}")

def construct_lagged_proxy_features(raw_ret: pd.DataFrame, proxy_groups: dict) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Construct leakage-safe U.S. proxy features.

    Rule:
    For any source-domain feature, first shift the source return by 1 day.
    Rolling features are then computed from this shifted series.

    This means Taiwan day t can use only source-domain data known by t-1.
    """
    features = pd.DataFrame(index=raw_ret.index)
    meta_rows = []

    source_groups = ["us_broad", "us_semi", "us_risk", "fx", "asia"]

    for group in source_groups:
        cols = proxy_groups.get(group, [])
        for col in cols:
            s0 = pd.to_numeric(raw_ret[col], errors="coerce")
            s = s0.shift(1)

            base = clean_colname(col)
            prefix = f"USPROXY_{group}_{base}"

            features[f"{prefix}_lag1"] = s

            for lag in LAG_WINDOWS:
                features[f"{prefix}_lag{lag}"] = s0.shift(lag)
                meta_rows.append({
                    "feature": f"{prefix}_lag{lag}",
                    "source_group": group,
                    "source_column": col,
                    "transformation": f"return_lag_{lag}",
                    "leakage_rule": "uses source return shifted by lag >= 1",
                })

            for w in ROLL_WINDOWS:
                features[f"{prefix}_roll_mean_{w}"] = s.rolling(w, min_periods=max(2, w // 2)).mean()
                features[f"{prefix}_roll_std_{w}"] = s.rolling(w, min_periods=max(2, w // 2)).std()
                features[f"{prefix}_roll_sum_{w}"] = s.rolling(w, min_periods=max(2, w // 2)).sum()
                features[f"{prefix}_roll_posrate_{w}"] = (s > 0).astype(float).rolling(w, min_periods=max(2, w // 2)).mean()

                meta_rows.extend([
                    {
                        "feature": f"{prefix}_roll_mean_{w}",
                        "source_group": group,
                        "source_column": col,
                        "transformation": f"shift1_rolling_mean_{w}",
                        "leakage_rule": "source series shifted by 1 before rolling",
                    },
                    {
                        "feature": f"{prefix}_roll_std_{w}",
                        "source_group": group,
                        "source_column": col,
                        "transformation": f"shift1_rolling_std_{w}",
                        "leakage_rule": "source series shifted by 1 before rolling",
                    },
                    {
                        "feature": f"{prefix}_roll_sum_{w}",
                        "source_group": group,
                        "source_column": col,
                        "transformation": f"shift1_rolling_sum_{w}",
                        "leakage_rule": "source series shifted by 1 before rolling",
                    },
                    {
                        "feature": f"{prefix}_roll_posrate_{w}",
                        "source_group": group,
                        "source_column": col,
                        "transformation": f"shift1_rolling_positive_rate_{w}",
                        "leakage_rule": "source series shifted by 1 before rolling",
                    },
                ])

    meta = pd.DataFrame(meta_rows).drop_duplicates()
    features = features.replace([np.inf, -np.inf], np.nan)
    return features, meta

proxy_features, proxy_feature_meta = construct_lagged_proxy_features(raw_ret, proxy_groups_raw)

print("Constructed proxy feature matrix:", proxy_features.shape)
print("Proxy feature meta:", proxy_feature_meta.shape)

save_parquet_csv(
    proxy_features,
    DATA_RUN_DIR / "notebook15_lagged_us_proxy_features",
    index=True,
)

save_table(
    proxy_feature_meta,
    "notebook15_03_lagged_proxy_feature_metadata.csv",
    f"table_15_03_lagged_proxy_feature_metadata_{RUN_ID}.csv",
)

# ============================================================
# 6. Build final modeling matrix and feature sets
# ============================================================

print("\n" + "=" * 100)
print("Step 3: Build final feature matrix and feature sets")
print("=" * 100)

target_cols_all = list(TARGET_COLS.values())
target_cols_set = set(target_cols_all)

# Remove target-like and future-return-like columns from features.
def is_forbidden_feature_col(c: str):
    lc = c.lower()
    if c in target_cols_set:
        return True
    if any(k in lc for k in ["target", "label", "future", "forward"]):
        return True
    if "regime" in lc and any(str(h) in lc for h in HORIZONS):
        return True
    return False

all_existing_feature_cols = [
    c for c in model_df.columns
    if not is_forbidden_feature_col(c)
    and pd.api.types.is_numeric_dtype(model_df[c])
]

def is_us_proxy_like(c: str):
    lc = c.lower()
    tokens = [
        "sp500", "s_p500", "spx", "spy", "nasdaq", "ixic", "ndx", "qqq",
        "soxx", "smh", "semiconductor", "semi", "vix", "us10", "10y",
        "yield", "treasury", "usdtwd", "usd_twd", "usd", "dxy",
        "nikkei", "n225", "hang", "hsi", "kospi",
        "usproxy",
    ]
    return any(t in lc for t in tokens)

def is_tw_local_like(c: str):
    lc = c.lower()
    tokens = ["0050", "006208", "00692", "00881", "taiex", "twii", "twse", "taiwan"]
    return any(t in lc for t in tokens)

# TW-only baseline:
# Use existing features that are clearly Taiwan/ETF-local, and exclude US/global proxy-like columns.
tw_only_existing_cols = [
    c for c in all_existing_feature_cols
    if is_tw_local_like(c) and not is_us_proxy_like(c)
]

# If too few TW-only features are detected, use all existing non-proxy features.
if len(tw_only_existing_cols) < 20:
    tw_only_existing_cols = [
        c for c in all_existing_feature_cols
        if not is_us_proxy_like(c)
    ]

# Existing all features reference.
existing_all_cols = all_existing_feature_cols

# Proxy feature groups.
proxy_cols_by_group = {
    "us_broad": proxy_feature_meta.loc[
        proxy_feature_meta["source_group"] == "us_broad", "feature"
    ].drop_duplicates().tolist() if len(proxy_feature_meta) else [],
    "us_semi": proxy_feature_meta.loc[
        proxy_feature_meta["source_group"] == "us_semi", "feature"
    ].drop_duplicates().tolist() if len(proxy_feature_meta) else [],
    "us_risk_fx": proxy_feature_meta.loc[
        proxy_feature_meta["source_group"].isin(["us_risk", "fx"]), "feature"
    ].drop_duplicates().tolist() if len(proxy_feature_meta) else [],
    "asia": proxy_feature_meta.loc[
        proxy_feature_meta["source_group"] == "asia", "feature"
    ].drop_duplicates().tolist() if len(proxy_feature_meta) else [],
}

proxy_all_cols = sorted(set(sum(proxy_cols_by_group.values(), [])))

# Combine model df with proxy features.
combined_df = model_df.join(proxy_features, how="left")

feature_sets = {
    "FS0_TW_only": tw_only_existing_cols,
    "FS1_TW_plus_US_broad": tw_only_existing_cols + proxy_cols_by_group["us_broad"],
    "FS2_TW_plus_US_semi": tw_only_existing_cols + proxy_cols_by_group["us_semi"],
    "FS3_TW_plus_US_risk_fx": tw_only_existing_cols + proxy_cols_by_group["us_risk_fx"],
    "FS4_TW_plus_all_US_proxy": tw_only_existing_cols + proxy_all_cols,
    "FS5_existing_all_reference": existing_all_cols,
}

# Remove empty and duplicate columns.
for fs_name, cols in list(feature_sets.items()):
    clean_cols = []
    seen = set()
    for c in cols:
        if c in combined_df.columns and c not in seen:
            clean_cols.append(c)
            seen.add(c)
    feature_sets[fs_name] = clean_cols

feature_set_audit_rows = []
for fs_name, cols in feature_sets.items():
    feature_set_audit_rows.append({
        "feature_set": fs_name,
        "n_features": len(cols),
        "n_existing_features": int(sum(c in all_existing_feature_cols for c in cols)),
        "n_proxy_features": int(sum(c in proxy_all_cols for c in cols)),
        "contains_us_broad": int(any(c in proxy_cols_by_group["us_broad"] for c in cols)),
        "contains_us_semi": int(any(c in proxy_cols_by_group["us_semi"] for c in cols)),
        "contains_us_risk_fx": int(any(c in proxy_cols_by_group["us_risk_fx"] for c in cols)),
        "contains_asia": int(any(c in proxy_cols_by_group["asia"] for c in cols)),
    })

feature_set_audit = pd.DataFrame(feature_set_audit_rows)

save_table(
    feature_set_audit,
    "notebook15_04_feature_set_audit.csv",
    f"table_15_04_feature_set_audit_{RUN_ID}.csv",
)

print("Feature set audit:")
print(feature_set_audit.to_string(index=False))

# Save feature list per set.
feature_list_rows = []
for fs_name, cols in feature_sets.items():
    for c in cols:
        feature_list_rows.append({
            "feature_set": fs_name,
            "feature": c,
            "is_proxy_feature": c in proxy_all_cols,
        })

feature_list_df = pd.DataFrame(feature_list_rows)

save_table(
    feature_list_df,
    "notebook15_05_feature_list_by_set.csv",
    f"table_15_05_feature_list_by_set_{RUN_ID}.csv",
)

save_parquet_csv(
    combined_df,
    DATA_RUN_DIR / "notebook15_combined_modeling_matrix_with_lagged_us_proxies",
    index=True,
)

# ============================================================
# 7. Purged / embargoed walk-forward split construction
# ============================================================

print("\n" + "=" * 100)
print("Step 4: Build purged / embargoed walk-forward folds")
print("=" * 100)

def build_predefined_folds(index: pd.DatetimeIndex, horizon: int):
    """
    Approximate the established AURORA / ROMA purged walk-forward split design.

    For 20d:
        train sizes: 694, 820, 883
        validation size: 169
        test size: 169, 169, 170
        embargo: 20
    For 60d:
        train sizes: 694, 820, 883
        validation size: 129
        test size: 129, 129, 130
        embargo: 60
    """
    n = len(index)

    if horizon == 20:
        specs = [
            {"fold_id": "WF1", "train_end": 694, "val_size": 169, "test_size": 169, "embargo": 20},
            {"fold_id": "WF2", "train_end": 820, "val_size": 169, "test_size": 169, "embargo": 20},
            {"fold_id": "WF3", "train_end": 883, "val_size": 169, "test_size": 170, "embargo": 20},
        ]
    elif horizon == 60:
        specs = [
            {"fold_id": "WF1", "train_end": 694, "val_size": 129, "test_size": 129, "embargo": 60},
            {"fold_id": "WF2", "train_end": 820, "val_size": 129, "test_size": 129, "embargo": 60},
            {"fold_id": "WF3", "train_end": 883, "val_size": 129, "test_size": 130, "embargo": 60},
        ]
    else:
        raise ValueError(horizon)

    folds = []

    for spec in specs:
        train_start = 0
        train_end = spec["train_end"]

        val_start = train_end + spec["embargo"]
        val_end = val_start + spec["val_size"]

        test_start = val_end + spec["embargo"]
        test_end = test_start + spec["test_size"]

        if test_end > n:
            continue

        fold = {
            "horizon": horizon,
            "fold_id": spec["fold_id"],
            "embargo": spec["embargo"],
            "train_idx": np.arange(train_start, train_end),
            "val_idx": np.arange(val_start, val_end),
            "test_idx": np.arange(test_start, test_end),
        }
        folds.append(fold)

    if len(folds) == 3:
        return folds

    print(f"Predefined folds unavailable for horizon={horizon}; using adaptive folds.")

    # Adaptive fallback.
    folds = []
    embargo = horizon
    val_size = 169 if horizon == 20 else 129
    test_size = 169 if horizon == 20 else 129
    train_ends = [
        int(n * 0.55),
        int(n * 0.65),
        int(n * 0.70),
    ]

    for i, train_end in enumerate(train_ends, start=1):
        val_start = train_end + embargo
        val_end = val_start + val_size
        test_start = val_end + embargo
        test_end = min(test_start + test_size, n)

        if test_end - test_start < max(60, test_size // 2):
            continue

        folds.append({
            "horizon": horizon,
            "fold_id": f"WF{i}",
            "embargo": embargo,
            "train_idx": np.arange(0, train_end),
            "val_idx": np.arange(val_start, val_end),
            "test_idx": np.arange(test_start, test_end),
        })

    return folds

all_folds = {}
fold_rows = []

for h in HORIZONS:
    folds = build_predefined_folds(combined_df.index, h)
    all_folds[h] = folds

    for f in folds:
        train_dates = combined_df.index[f["train_idx"]]
        val_dates = combined_df.index[f["val_idx"]]
        test_dates = combined_df.index[f["test_idx"]]

        fold_rows.append({
            "horizon": h,
            "fold_id": f["fold_id"],
            "embargo": f["embargo"],
            "train_n": len(f["train_idx"]),
            "val_n": len(f["val_idx"]),
            "test_n": len(f["test_idx"]),
            "train_start": train_dates.min(),
            "train_end": train_dates.max(),
            "val_start": val_dates.min(),
            "val_end": val_dates.max(),
            "test_start": test_dates.min(),
            "test_end": test_dates.max(),
        })

fold_table = pd.DataFrame(fold_rows)

save_table(
    fold_table,
    "notebook15_06_purged_walk_forward_fold_table.csv",
    f"table_15_06_purged_walk_forward_fold_table_{RUN_ID}.csv",
)

print("Fold table:")
print(fold_table.to_string(index=False))

# ============================================================
# 8. Model zoo
# ============================================================

print("\n" + "=" * 100)
print("Step 5: Define model zoo")
print("=" * 100)

def make_model_zoo():
    models = {}

    models["M1_logistic_balanced"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            multi_class="auto",
            solver="lbfgs",
            random_state=RANDOM_STATE,
        )),
    ])

    models["M2_random_forest_balanced"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", RandomForestClassifier(
            n_estimators=300,
            max_depth=6,
            min_samples_leaf=10,
            class_weight="balanced_subsample",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])

    models["M3_extra_trees_balanced"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", ExtraTreesClassifier(
            n_estimators=400,
            max_depth=6,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ])

    models["M4_hist_gradient_boosting"] = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("clf", HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.04,
            max_leaf_nodes=15,
            l2_regularization=0.1,
            random_state=RANDOM_STATE,
        )),
    ])

    if HAS_LGBM:
        models["M5_lightgbm_balanced"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", LGBMClassifier(
                n_estimators=300,
                learning_rate=0.03,
                num_leaves=15,
                max_depth=5,
                min_child_samples=20,
                subsample=0.8,
                colsample_bytree=0.8,
                class_weight="balanced",
                random_state=RANDOM_STATE,
                n_jobs=-1,
                verbose=-1,
            )),
        ])

    if HAS_XGB:
        models["M6_xgboost_multiclass"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("clf", XGBClassifier(
                n_estimators=250,
                learning_rate=0.03,
                max_depth=4,
                min_child_weight=5,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="multi:softprob",
                eval_metric="mlogloss",
                random_state=RANDOM_STATE,
                n_jobs=-1,
            )),
        ])

    return models

MODEL_ZOO = make_model_zoo()

model_zoo_table = pd.DataFrame([
    {
        "model_name": name,
        "model_repr": repr(model),
    }
    for name, model in MODEL_ZOO.items()
])

save_table(
    model_zoo_table,
    "notebook15_07_model_zoo.csv",
    f"table_15_07_model_zoo_{RUN_ID}.csv",
)

print("Model zoo:")
for name in MODEL_ZOO:
    print(" -", name)

# ============================================================
# 9. Forecasting experiment
# ============================================================

print("\n" + "=" * 100)
print("Step 6: Run purged walk-forward forecasting experiment")
print("=" * 100)

def fit_predict_one_fold(
    df: pd.DataFrame,
    target_col: str,
    feature_cols: list[str],
    fold: dict,
    model_name: str,
    model_template,
):
    train_idx = fold["train_idx"]
    val_idx = fold["val_idx"]
    test_idx = fold["test_idx"]

    cols = feature_cols + [target_col]
    sub = df.iloc[np.concatenate([train_idx, val_idx, test_idx])][cols].copy()

    # Valid rows must have target.
    y_all = pd.to_numeric(df[target_col], errors="coerce")

    X_train = df.iloc[train_idx][feature_cols]
    y_train = y_all.iloc[train_idx].astype("Int64")

    X_val = df.iloc[val_idx][feature_cols]
    y_val = y_all.iloc[val_idx].astype("Int64")

    X_test = df.iloc[test_idx][feature_cols]
    y_test = y_all.iloc[test_idx].astype("Int64")

    train_mask = y_train.notna()
    val_mask = y_val.notna()
    test_mask = y_test.notna()

    X_train = X_train.loc[train_mask]
    y_train = y_train.loc[train_mask].astype(int)

    X_val = X_val.loc[val_mask]
    y_val = y_val.loc[val_mask].astype(int)

    X_test = X_test.loc[test_mask]
    y_test = y_test.loc[test_mask].astype(int)

    # Remove rows where all features are missing.
    train_feature_mask = ~X_train.isna().all(axis=1)
    val_feature_mask = ~X_val.isna().all(axis=1)
    test_feature_mask = ~X_test.isna().all(axis=1)

    X_train = X_train.loc[train_feature_mask]
    y_train = y_train.loc[train_feature_mask]

    X_val = X_val.loc[val_feature_mask]
    y_val = y_val.loc[val_feature_mask]

    X_test = X_test.loc[test_feature_mask]
    y_test = y_test.loc[test_feature_mask]

    if len(X_train) < 50 or len(np.unique(y_train)) < 2:
        raise ValueError("Insufficient train data or fewer than two classes.")

    model = clone(model_template)
    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)
    test_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        val_proba_raw = model.predict_proba(X_val)
        test_proba_raw = model.predict_proba(X_test)

        clf = model.named_steps["clf"] if isinstance(model, Pipeline) and "clf" in model.named_steps else model
        model_classes = getattr(clf, "classes_", ORDINAL_CLASSES)

        val_proba = align_proba_to_classes(model_classes, val_proba_raw)
        test_proba = align_proba_to_classes(model_classes, test_proba_raw)
    else:
        val_proba = None
        test_proba = None

    val_metrics = evaluate_classification(y_val, val_pred, val_proba)
    test_metrics = evaluate_classification(y_test, test_pred, test_proba)

    val_pred_df = pd.DataFrame({
        "date": X_val.index,
        "y_true": y_val.values,
        "y_pred": val_pred,
        "split": "validation",
    })

    test_pred_df = pd.DataFrame({
        "date": X_test.index,
        "y_true": y_test.values,
        "y_pred": test_pred,
        "split": "test",
    })

    if val_proba is not None:
        for i, cls in enumerate(ORDINAL_CLASSES):
            val_pred_df[f"proba_class_{cls}"] = val_proba[:, i]
            test_pred_df[f"proba_class_{cls}"] = test_proba[:, i]

    return model, val_metrics, test_metrics, val_pred_df, test_pred_df

result_rows = []
prediction_frames = []

experiment_errors = []

for horizon in HORIZONS:
    target_col = TARGET_COLS[horizon]
    folds = all_folds[horizon]

    for feature_set_name, feature_cols in feature_sets.items():
        if len(feature_cols) == 0:
            print(f"Skipping empty feature set: {feature_set_name}")
            continue

        for model_name, model_template in MODEL_ZOO.items():
            for fold in folds:
                print(
                    f"H={horizon} | {fold['fold_id']} | "
                    f"{feature_set_name} | {model_name} | n_features={len(feature_cols)}"
                )

                try:
                    model, val_metrics, test_metrics, val_pred_df, test_pred_df = fit_predict_one_fold(
                        df=combined_df,
                        target_col=target_col,
                        feature_cols=feature_cols,
                        fold=fold,
                        model_name=model_name,
                        model_template=model_template,
                    )

                    base_meta = {
                        "horizon": horizon,
                        "target_column": target_col,
                        "fold_id": fold["fold_id"],
                        "feature_set": feature_set_name,
                        "n_features": len(feature_cols),
                        "model_name": model_name,
                    }

                    result_rows.append({
                        **base_meta,
                        "split": "validation",
                        **val_metrics,
                    })

                    result_rows.append({
                        **base_meta,
                        "split": "test",
                        **test_metrics,
                    })

                    val_pred_df = val_pred_df.assign(**base_meta)
                    test_pred_df = test_pred_df.assign(**base_meta)

                    prediction_frames.append(val_pred_df)
                    prediction_frames.append(test_pred_df)

                except Exception as e:
                    experiment_errors.append({
                        "horizon": horizon,
                        "fold_id": fold["fold_id"],
                        "feature_set": feature_set_name,
                        "model_name": model_name,
                        "error": repr(e),
                    })
                    print("  ERROR:", repr(e))

results_raw = pd.DataFrame(result_rows)
predictions_all = pd.concat(prediction_frames, axis=0, ignore_index=True) if prediction_frames else pd.DataFrame()
errors_df = pd.DataFrame(experiment_errors)

save_table(
    results_raw,
    "notebook15_08_fold_level_results_raw.csv",
    f"table_15_08_fold_level_results_raw_{RUN_ID}.csv",
)

save_parquet_csv(
    predictions_all,
    PRED_RUN_DIR / "notebook15_all_fold_predictions",
    index=False,
)

save_table(
    errors_df,
    "notebook15_09_experiment_errors.csv",
    f"table_15_09_experiment_errors_{RUN_ID}.csv",
)

print("Fold-level results:", results_raw.shape)
print("Predictions:", predictions_all.shape)
print("Errors:", errors_df.shape)

# ============================================================
# 10. Aggregate metrics and select validation-best models
# ============================================================

print("\n" + "=" * 100)
print("Step 7: Aggregate metrics and select validation-best models")
print("=" * 100)

metric_cols = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "weighted_f1",
    "ordinal_mae",
    "quadratic_weighted_kappa",
    "log_loss",
    "brier",
    "ece",
    "ordinal_rmse_expected",
]

agg_rows = []

for keys, g in results_raw.groupby(["horizon", "feature_set", "model_name", "split"]):
    h, fs, model_name, split = keys
    row = {
        "horizon": h,
        "feature_set": fs,
        "model_name": model_name,
        "split": split,
        "n_folds": g["fold_id"].nunique(),
        "n_total": int(g["n"].sum()) if "n" in g.columns else np.nan,
    }
    for m in metric_cols:
        if m in g.columns:
            row[m] = float(g[m].mean())
            row[f"{m}_std"] = float(g[m].std(ddof=1)) if len(g) > 1 else 0.0
    agg_rows.append(row)

agg_results = pd.DataFrame(agg_rows)

agg_ranked = []
for (h, split), g in agg_results.groupby(["horizon", "split"]):
    gr = rank_composite(g)
    agg_ranked.append(gr)

agg_results_ranked = pd.concat(agg_ranked, axis=0, ignore_index=True) if agg_ranked else pd.DataFrame()

save_table(
    agg_results_ranked,
    "notebook15_10_aggregate_leaderboard_ranked.csv",
    f"table_15_10_aggregate_leaderboard_ranked_{RUN_ID}.csv",
)

# Select model per horizon + feature set using validation composite rank.
validation_ranked = agg_results_ranked[agg_results_ranked["split"] == "validation"].copy()
validation_ranked = validation_ranked.sort_values(
    ["horizon", "feature_set", "composite_rank", "macro_f1", "quadratic_weighted_kappa"],
    ascending=[True, True, True, False, False],
)

selected_rows = []
for (h, fs), g in validation_ranked.groupby(["horizon", "feature_set"]):
    selected_rows.append(g.iloc[0].to_dict())

selected_models = pd.DataFrame(selected_rows)

# Attach corresponding test performance.
test_ranked = agg_results_ranked[agg_results_ranked["split"] == "test"].copy()

selected_with_test_rows = []
for _, row in selected_models.iterrows():
    h = row["horizon"]
    fs = row["feature_set"]
    model_name = row["model_name"]

    test_match = test_ranked[
        (test_ranked["horizon"] == h)
        & (test_ranked["feature_set"] == fs)
        & (test_ranked["model_name"] == model_name)
    ]

    out = {
        "horizon": h,
        "feature_set": fs,
        "selected_model": model_name,
        "validation_composite_rank": row.get("composite_rank", np.nan),
        "validation_macro_f1": row.get("macro_f1", np.nan),
        "validation_qwk": row.get("quadratic_weighted_kappa", np.nan),
        "validation_ordinal_mae": row.get("ordinal_mae", np.nan),
        "validation_ece": row.get("ece", np.nan),
    }

    if len(test_match):
        tr = test_match.iloc[0]
        out.update({
            "test_accuracy": tr.get("accuracy", np.nan),
            "test_balanced_accuracy": tr.get("balanced_accuracy", np.nan),
            "test_macro_f1": tr.get("macro_f1", np.nan),
            "test_qwk": tr.get("quadratic_weighted_kappa", np.nan),
            "test_ordinal_mae": tr.get("ordinal_mae", np.nan),
            "test_log_loss": tr.get("log_loss", np.nan),
            "test_brier": tr.get("brier", np.nan),
            "test_ece": tr.get("ece", np.nan),
            "test_ordinal_rmse_expected": tr.get("ordinal_rmse_expected", np.nan),
        })

    selected_with_test_rows.append(out)

selected_summary = pd.DataFrame(selected_with_test_rows)
selected_summary = selected_summary.sort_values(["horizon", "feature_set"]).reset_index(drop=True)

save_table(
    selected_summary,
    "notebook15_11_validation_selected_feature_set_summary.csv",
    f"table_15_11_validation_selected_feature_set_summary_{RUN_ID}.csv",
)

print("Selected model summary:")
print(selected_summary.to_string(index=False))

# ============================================================
# 11. Incremental proxy-factor analysis
# ============================================================

print("\n" + "=" * 100)
print("Step 8: Incremental proxy-factor analysis")
print("=" * 100)

increment_rows = []

for h in HORIZONS:
    base = selected_summary[
        (selected_summary["horizon"] == h)
        & (selected_summary["feature_set"] == "FS0_TW_only")
    ]

    if base.empty:
        continue

    base = base.iloc[0]

    for _, row in selected_summary[selected_summary["horizon"] == h].iterrows():
        increment_rows.append({
            "horizon": h,
            "feature_set": row["feature_set"],
            "selected_model": row["selected_model"],
            "delta_test_accuracy_vs_TW_only": row["test_accuracy"] - base["test_accuracy"],
            "delta_test_balanced_accuracy_vs_TW_only": row["test_balanced_accuracy"] - base["test_balanced_accuracy"],
            "delta_test_macro_f1_vs_TW_only": row["test_macro_f1"] - base["test_macro_f1"],
            "delta_test_qwk_vs_TW_only": row["test_qwk"] - base["test_qwk"],
            "delta_test_ordinal_mae_vs_TW_only": row["test_ordinal_mae"] - base["test_ordinal_mae"],
            "delta_test_log_loss_vs_TW_only": row["test_log_loss"] - base["test_log_loss"],
            "delta_test_brier_vs_TW_only": row["test_brier"] - base["test_brier"],
            "delta_test_ece_vs_TW_only": row["test_ece"] - base["test_ece"],
            "base_test_macro_f1": base["test_macro_f1"],
            "base_test_qwk": base["test_qwk"],
            "base_test_ordinal_mae": base["test_ordinal_mae"],
            "feature_test_macro_f1": row["test_macro_f1"],
            "feature_test_qwk": row["test_qwk"],
            "feature_test_ordinal_mae": row["test_ordinal_mae"],
        })

incremental_summary = pd.DataFrame(increment_rows)

save_table(
    incremental_summary,
    "notebook15_12_incremental_proxy_factor_summary.csv",
    f"table_15_12_incremental_proxy_factor_summary_{RUN_ID}.csv",
)

print("Incremental proxy-factor summary:")
print(incremental_summary.to_string(index=False))

# ============================================================
# 12. Paired test-set comparison for selected models
# ============================================================

print("\n" + "=" * 100)
print("Step 9: Paired test-set comparison for selected models")
print("=" * 100)

def get_selected_prediction_frame(horizon, feature_set):
    sel = selected_summary[
        (selected_summary["horizon"] == horizon)
        & (selected_summary["feature_set"] == feature_set)
    ]
    if sel.empty:
        return pd.DataFrame()

    model_name = sel.iloc[0]["selected_model"]

    p = predictions_all[
        (predictions_all["horizon"] == horizon)
        & (predictions_all["feature_set"] == feature_set)
        & (predictions_all["model_name"] == model_name)
        & (predictions_all["split"] == "test")
    ].copy()

    p["date"] = pd.to_datetime(p["date"])
    return p

def paired_bootstrap_metric_diff(
    base_df: pd.DataFrame,
    alt_df: pd.DataFrame,
    metric_name: str,
    n_boot: int = 2000,
    block_size: int = 20,
    random_state: int = RANDOM_STATE,
):
    """
    Paired moving-block bootstrap over dates.

    Difference = alternative metric - base metric.
    For ordinal_mae and log_loss, negative is better.
    For accuracy, macro_f1, qwk, positive is better.
    """
    rng = np.random.default_rng(random_state)

    common_dates = sorted(set(base_df["date"]).intersection(set(alt_df["date"])))
    if not common_dates:
        return None

    b = base_df.set_index("date").loc[common_dates].sort_index()
    a = alt_df.set_index("date").loc[common_dates].sort_index()

    y = b["y_true"].astype(int).values
    b_pred = b["y_pred"].astype(int).values
    a_pred = a["y_pred"].astype(int).values

    proba_cols = [f"proba_class_{c}" for c in ORDINAL_CLASSES]
    has_proba = all(c in b.columns for c in proba_cols) and all(c in a.columns for c in proba_cols)

    if has_proba:
        b_proba = b[proba_cols].values
        a_proba = a[proba_cols].values
    else:
        b_proba = None
        a_proba = None

    def metric(y_sub, pred_sub, proba_sub):
        if metric_name == "accuracy":
            return accuracy_score(y_sub, pred_sub)
        if metric_name == "macro_f1":
            return f1_score(y_sub, pred_sub, average="macro", zero_division=0)
        if metric_name == "qwk":
            return cohen_kappa_score(y_sub, pred_sub, weights="quadratic")
        if metric_name == "ordinal_mae":
            return ordinal_mae(y_sub, pred_sub)
        if metric_name == "log_loss":
            if proba_sub is None:
                return np.nan
            return log_loss(y_sub, proba_sub, labels=list(ORDINAL_CLASSES))
        if metric_name == "brier":
            if proba_sub is None:
                return np.nan
            return brier_multiclass(y_sub, proba_sub)
        raise ValueError(metric_name)

    observed = metric(y, a_pred, a_proba) - metric(y, b_pred, b_proba)

    n = len(y)
    starts = np.arange(0, n)
    boot_vals = []

    for _ in range(n_boot):
        idxs = []
        while len(idxs) < n:
            s = int(rng.choice(starts))
            block = [(s + j) % n for j in range(block_size)]
            idxs.extend(block)
        idxs = np.array(idxs[:n])

        y_s = y[idxs]
        b_pred_s = b_pred[idxs]
        a_pred_s = a_pred[idxs]

        if has_proba:
            b_proba_s = b_proba[idxs]
            a_proba_s = a_proba[idxs]
        else:
            b_proba_s = None
            a_proba_s = None

        val = metric(y_s, a_pred_s, a_proba_s) - metric(y_s, b_pred_s, b_proba_s)
        boot_vals.append(val)

    boot_vals = np.asarray(boot_vals)
    ci_low, ci_high = np.nanpercentile(boot_vals, [2.5, 97.5])

    if metric_name in ["ordinal_mae", "log_loss", "brier"]:
        prob_improve = float(np.nanmean(boot_vals < 0))
        significant_improve = bool(ci_high < 0)
    else:
        prob_improve = float(np.nanmean(boot_vals > 0))
        significant_improve = bool(ci_low > 0)

    return {
        "metric": metric_name,
        "n_dates": int(n),
        "observed_alt_minus_base": float(observed),
        "ci_low": float(ci_low),
        "ci_high": float(ci_high),
        "prob_improvement": prob_improve,
        "significant_improvement_95": significant_improve,
        "n_boot": int(n_boot),
        "block_size": int(block_size),
    }

paired_rows = []

for h in HORIZONS:
    base_pred = get_selected_prediction_frame(h, "FS0_TW_only")
    if base_pred.empty:
        continue

    for fs in feature_sets.keys():
        if fs == "FS0_TW_only":
            continue

        alt_pred = get_selected_prediction_frame(h, fs)
        if alt_pred.empty:
            continue

        for metric_name in ["accuracy", "macro_f1", "qwk", "ordinal_mae", "log_loss", "brier"]:
            try:
                res = paired_bootstrap_metric_diff(
                    base_df=base_pred,
                    alt_df=alt_pred,
                    metric_name=metric_name,
                    n_boot=2000,
                    block_size=20 if h == 20 else 60,
                )
                if res is None:
                    continue
                paired_rows.append({
                    "horizon": h,
                    "base_feature_set": "FS0_TW_only",
                    "alt_feature_set": fs,
                    **res,
                })
            except Exception as e:
                paired_rows.append({
                    "horizon": h,
                    "base_feature_set": "FS0_TW_only",
                    "alt_feature_set": fs,
                    "metric": metric_name,
                    "error": repr(e),
                })

paired_bootstrap = pd.DataFrame(paired_rows)

save_table(
    paired_bootstrap,
    "notebook15_13_paired_block_bootstrap_feature_set_comparison.csv",
    f"table_15_13_paired_block_bootstrap_feature_set_comparison_{RUN_ID}.csv",
)

print("Paired bootstrap comparison:")
print(paired_bootstrap.head(80).to_string(index=False))

# ============================================================
# 13. Selected probability exports for downstream allocation
# ============================================================

print("\n" + "=" * 100)
print("Step 10: Export selected probability panels for downstream experiments")
print("=" * 100)

selected_probability_rows = []

for h in HORIZONS:
    # Pick the validation-selected best feature set by validation composite rank,
    # excluding existing-all reference if a proxy feature set is available.
    candidates = selected_summary[selected_summary["horizon"] == h].copy()

    preferred = candidates[
        candidates["feature_set"].isin([
            "FS1_TW_plus_US_broad",
            "FS2_TW_plus_US_semi",
            "FS3_TW_plus_US_risk_fx",
            "FS4_TW_plus_all_US_proxy",
            "FS0_TW_only",
        ])
    ].copy()

    if preferred.empty:
        preferred = candidates

    # Selection objective:
    # First by validation composite rank, then by test is NOT used.
    preferred = preferred.sort_values(
        ["validation_composite_rank", "validation_macro_f1", "validation_qwk"],
        ascending=[True, False, False],
    )

    if preferred.empty:
        continue

    chosen = preferred.iloc[0]
    fs = chosen["feature_set"]
    model_name = chosen["selected_model"]

    pred = predictions_all[
        (predictions_all["horizon"] == h)
        & (predictions_all["feature_set"] == fs)
        & (predictions_all["model_name"] == model_name)
        & (predictions_all["split"] == "test")
    ].copy()

    if pred.empty:
        continue

    pred["date"] = pd.to_datetime(pred["date"])
    pred = pred.sort_values("date")

    proba_cols = [f"proba_class_{c}" for c in ORDINAL_CLASSES]
    export_cols = ["date", "y_true", "y_pred"] + [c for c in proba_cols if c in pred.columns]

    export = pred[export_cols].copy()
    export["horizon"] = h
    export["selected_feature_set"] = fs
    export["selected_model"] = model_name
    export["probability_source"] = f"Notebook15_{fs}_{model_name}_{h}d"

    base = PROBA_RUN_DIR / f"notebook15_selected_probabilities_{h}d_{fs}_{model_name}"
    save_parquet_csv(export, base, index=False)

    selected_probability_rows.append({
        "horizon": h,
        "selected_feature_set": fs,
        "selected_model": model_name,
        "probability_source": f"Notebook15_{fs}_{model_name}_{h}d",
        "n_rows": len(export),
        "start_date": export["date"].min(),
        "end_date": export["date"].max(),
        "path_csv": str(Path(str(base) + ".csv")),
        "path_parquet": str(Path(str(base) + ".parquet")),
    })

selected_probability_index = pd.DataFrame(selected_probability_rows)

save_table(
    selected_probability_index,
    "notebook15_14_selected_probability_export_index.csv",
    f"table_15_14_selected_probability_export_index_{RUN_ID}.csv",
)

print("Selected probability export index:")
print(selected_probability_index.to_string(index=False))

# ============================================================
# 14. Paper-ready tables
# ============================================================

print("\n" + "=" * 100)
print("Step 11: Build paper-ready tables")
print("=" * 100)

paper_selected = selected_summary.copy()
round_cols = [
    "validation_macro_f1",
    "validation_qwk",
    "validation_ordinal_mae",
    "validation_ece",
    "test_accuracy",
    "test_balanced_accuracy",
    "test_macro_f1",
    "test_qwk",
    "test_ordinal_mae",
    "test_log_loss",
    "test_brier",
    "test_ece",
]

for c in round_cols:
    if c in paper_selected.columns:
        paper_selected[c] = pd.to_numeric(paper_selected[c], errors="coerce").round(4)

paper_incremental = incremental_summary.copy()
for c in paper_incremental.columns:
    if c.startswith("delta_") or c.startswith("base_") or c.startswith("feature_"):
        paper_incremental[c] = pd.to_numeric(paper_incremental[c], errors="coerce").round(4)

paper_bootstrap = paired_bootstrap.copy()
for c in ["observed_alt_minus_base", "ci_low", "ci_high", "prob_improvement"]:
    if c in paper_bootstrap.columns:
        paper_bootstrap[c] = pd.to_numeric(paper_bootstrap[c], errors="coerce").round(4)

save_table(
    paper_selected,
    "notebook15_15_paper_selected_feature_set_forecasting_table.csv",
    f"table_15_15_paper_selected_feature_set_forecasting_table_{RUN_ID}.csv",
)

save_table(
    paper_incremental,
    "notebook15_16_paper_incremental_proxy_factor_table.csv",
    f"table_15_16_paper_incremental_proxy_factor_table_{RUN_ID}.csv",
)

save_table(
    paper_bootstrap,
    "notebook15_17_paper_paired_bootstrap_proxy_factor_table.csv",
    f"table_15_17_paper_paired_bootstrap_proxy_factor_table_{RUN_ID}.csv",
)

print("Paper selected forecasting table:")
print(paper_selected.to_string(index=False))

print("\nPaper incremental table:")
print(paper_incremental.to_string(index=False))

# ============================================================
# 15. Figures
# ============================================================

print("\n" + "=" * 100)
print("Step 12: Create figures")
print("=" * 100)

figure_records = []

def record_figure(path: Path, figure_id: str, title: str, caption: str):
    figure_records.append({
        "figure_id": figure_id,
        "path": str(path),
        "title": title,
        "caption": caption,
    })

# Figure 15.1: Feature-set macro F1 by horizon.
plot_df = paper_selected.copy()

if len(plot_df):
    for metric, ylabel, fname, figid in [
        ("test_macro_f1", "Test macro F1", "figure15_01_test_macro_f1_by_feature_set.png", "Figure 15.1"),
        ("test_qwk", "Test quadratic weighted kappa", "figure15_02_test_qwk_by_feature_set.png", "Figure 15.2"),
        ("test_ordinal_mae", "Test ordinal MAE", "figure15_03_test_ordinal_mae_by_feature_set.png", "Figure 15.3"),
        ("test_ece", "Test ECE", "figure15_04_test_ece_by_feature_set.png", "Figure 15.4"),
    ]:
        if metric not in plot_df.columns:
            continue

        plt.figure(figsize=(11, 5))
        if HAS_SEABORN:
            sns.barplot(
                data=plot_df,
                x="feature_set",
                y=metric,
                hue="horizon",
            )
            plt.legend(title="Horizon")
        else:
            for h in HORIZONS:
                sub = plot_df[plot_df["horizon"] == h]
                plt.plot(sub["feature_set"], sub[metric], marker="o", label=f"{h}d")
            plt.legend()

        plt.title(ylabel + " by feature set")
        plt.xlabel("Feature set")
        plt.ylabel(ylabel)
        plt.xticks(rotation=35, ha="right")
        plt.grid(axis="y", alpha=0.3)
        plt.tight_layout()

        fig_path = PAPER_FIGURE_DIR / fname
        plt.savefig(fig_path, dpi=220)
        plt.close()

        record_figure(
            fig_path,
            figid,
            ylabel + " by feature set",
            f"{ylabel} of validation-selected models across feature sets and horizons.",
        )

# Figure 15.5: Incremental macro F1 vs TW-only.
if len(paper_incremental):
    plt.figure(figsize=(11, 5))
    plot_inc = paper_incremental[paper_incremental["feature_set"] != "FS0_TW_only"].copy()

    if "delta_test_macro_f1_vs_TW_only" in plot_inc.columns:
        if HAS_SEABORN:
            sns.barplot(
                data=plot_inc,
                x="feature_set",
                y="delta_test_macro_f1_vs_TW_only",
                hue="horizon",
            )
            plt.legend(title="Horizon")
        else:
            for h in HORIZONS:
                sub = plot_inc[plot_inc["horizon"] == h]
                plt.plot(sub["feature_set"], sub["delta_test_macro_f1_vs_TW_only"], marker="o", label=f"{h}d")
            plt.legend()

        plt.axhline(0, color="black", linewidth=1)
        plt.title("Incremental macro F1 relative to TW-only")
        plt.xlabel("Feature set")
        plt.ylabel("Δ test macro F1")
        plt.xticks(rotation=35, ha="right")
        plt.grid(axis="y", alpha=0.3)
        plt.tight_layout()

        fig_path = PAPER_FIGURE_DIR / "figure15_05_delta_macro_f1_vs_tw_only.png"
        plt.savefig(fig_path, dpi=220)
        plt.close()

        record_figure(
            fig_path,
            "Figure 15.5",
            "Incremental macro F1 relative to TW-only",
            "Change in test macro F1 from adding lagged U.S. proxy factors relative to the Taiwan-only feature set.",
        )

# Figure 15.6: Paired bootstrap heatmap for macro F1.
if len(paper_bootstrap):
    metric = "macro_f1"
    sub = paper_bootstrap[paper_bootstrap["metric"] == metric].copy()

    if len(sub):
        pivot = sub.pivot_table(
            index="alt_feature_set",
            columns="horizon",
            values="observed_alt_minus_base",
            aggfunc="first",
        )

        plt.figure(figsize=(7, 5))
        if HAS_SEABORN:
            sns.heatmap(pivot, annot=True, fmt=".3f", cmap="RdBu", center=0)
        else:
            plt.imshow(pivot.values, aspect="auto")
            plt.colorbar()
            plt.xticks(range(len(pivot.columns)), pivot.columns)
            plt.yticks(range(len(pivot.index)), pivot.index)

        plt.title("Paired Δ macro F1 vs TW-only")
        plt.xlabel("Horizon")
        plt.ylabel("Alternative feature set")
        plt.tight_layout()

        fig_path = PAPER_FIGURE_DIR / "figure15_06_paired_delta_macro_f1_heatmap.png"
        plt.savefig(fig_path, dpi=220)
        plt.close()

        record_figure(
            fig_path,
            "Figure 15.6",
            "Paired delta macro F1 versus TW-only",
            "Observed paired difference in macro F1 between each proxy-factor feature set and the Taiwan-only baseline.",
        )

# Figure 15.7: Feature count composition.
if len(feature_set_audit):
    plt.figure(figsize=(10, 5))
    x = np.arange(len(feature_set_audit))
    plt.bar(x, feature_set_audit["n_existing_features"], label="Existing/TW features")
    plt.bar(
        x,
        feature_set_audit["n_proxy_features"],
        bottom=feature_set_audit["n_existing_features"],
        label="Lagged U.S. proxy features",
    )
    plt.xticks(x, feature_set_audit["feature_set"], rotation=35, ha="right")
    plt.ylabel("Number of features")
    plt.title("Feature-set composition")
    plt.legend()
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()

    fig_path = PAPER_FIGURE_DIR / "figure15_07_feature_set_composition.png"
    plt.savefig(fig_path, dpi=220)
    plt.close()

    record_figure(
        fig_path,
        "Figure 15.7",
        "Feature-set composition",
        "Number of existing Taiwan/local features and lagged U.S. proxy features in each feature set.",
    )

figure_index = pd.DataFrame(figure_records)

save_table(
    figure_index,
    "notebook15_18_figure_index.csv",
    f"table_15_18_figure_index_{RUN_ID}.csv",
)

print("Figure index:")
print(figure_index.to_string(index=False))

# ============================================================
# 16. Claim checklist
# ============================================================

print("\n" + "=" * 100)
print("Step 13: Claim checklist")
print("=" * 100)

claim_rows = []

for h in HORIZONS:
    inc_h = incremental_summary[incremental_summary["horizon"] == h].copy()

    if inc_h.empty:
        continue

    proxy_inc = inc_h[inc_h["feature_set"] != "FS0_TW_only"].copy()

    best_macro = proxy_inc.sort_values("delta_test_macro_f1_vs_TW_only", ascending=False).head(1)
    best_qwk = proxy_inc.sort_values("delta_test_qwk_vs_TW_only", ascending=False).head(1)
    best_mae = proxy_inc.sort_values("delta_test_ordinal_mae_vs_TW_only", ascending=True).head(1)

    claim_rows.append({
        "claim_id": f"C15_{h}d_macro_f1_best_proxy",
        "horizon": h,
        "claim": "Best proxy feature set improves test macro F1 relative to TW-only.",
        "support_value": float(best_macro["delta_test_macro_f1_vs_TW_only"].iloc[0]) if len(best_macro) else np.nan,
        "support_feature_set": best_macro["feature_set"].iloc[0] if len(best_macro) else None,
        "claim_supported_directionally": bool(len(best_macro) and best_macro["delta_test_macro_f1_vs_TW_only"].iloc[0] > 0),
        "paper_safe_wording": (
            "Lagged U.S. proxy factors improved macro F1 directionally."
            if len(best_macro) and best_macro["delta_test_macro_f1_vs_TW_only"].iloc[0] > 0
            else "Lagged U.S. proxy factors did not improve macro F1 directionally."
        ),
    })

    claim_rows.append({
        "claim_id": f"C15_{h}d_qwk_best_proxy",
        "horizon": h,
        "claim": "Best proxy feature set improves test QWK relative to TW-only.",
        "support_value": float(best_qwk["delta_test_qwk_vs_TW_only"].iloc[0]) if len(best_qwk) else np.nan,
        "support_feature_set": best_qwk["feature_set"].iloc[0] if len(best_qwk) else None,
        "claim_supported_directionally": bool(len(best_qwk) and best_qwk["delta_test_qwk_vs_TW_only"].iloc[0] > 0),
        "paper_safe_wording": (
            "Lagged U.S. proxy factors improved QWK directionally."
            if len(best_qwk) and best_qwk["delta_test_qwk_vs_TW_only"].iloc[0] > 0
            else "Lagged U.S. proxy factors did not improve QWK directionally."
        ),
    })

    claim_rows.append({
        "claim_id": f"C15_{h}d_ordinal_mae_best_proxy",
        "horizon": h,
        "claim": "Best proxy feature set reduces test ordinal MAE relative to TW-only.",
        "support_value": float(best_mae["delta_test_ordinal_mae_vs_TW_only"].iloc[0]) if len(best_mae) else np.nan,
        "support_feature_set": best_mae["feature_set"].iloc[0] if len(best_mae) else None,
        "claim_supported_directionally": bool(len(best_mae) and best_mae["delta_test_ordinal_mae_vs_TW_only"].iloc[0] < 0),
        "paper_safe_wording": (
            "Lagged U.S. proxy factors reduced ordinal MAE directionally."
            if len(best_mae) and best_mae["delta_test_ordinal_mae_vs_TW_only"].iloc[0] < 0
            else "Lagged U.S. proxy factors did not reduce ordinal MAE directionally."
        ),
    })

# Bootstrap significance claims.
if len(paired_bootstrap):
    for _, r in paired_bootstrap.iterrows():
        if "significant_improvement_95" not in r:
            continue
        if pd.isna(r.get("significant_improvement_95", np.nan)):
            continue
        claim_rows.append({
            "claim_id": f"C15_bootstrap_{int(r['horizon'])}_{r['alt_feature_set']}_{r['metric']}",
            "horizon": int(r["horizon"]),
            "claim": f"{r['alt_feature_set']} significantly improves {r['metric']} vs TW-only at 95%.",
            "support_value": r.get("observed_alt_minus_base", np.nan),
            "support_feature_set": r["alt_feature_set"],
            "claim_supported_directionally": bool(r.get("significant_improvement_95", False)),
            "paper_safe_wording": (
                f"Paired block bootstrap supports a significant improvement for {r['metric']}."
                if bool(r.get("significant_improvement_95", False))
                else f"Paired block bootstrap does not support a significant improvement for {r['metric']}."
            ),
        })

claim_checklist = pd.DataFrame(claim_rows)

save_table(
    claim_checklist,
    "notebook15_19_claim_checklist.csv",
    f"table_15_19_claim_checklist_{RUN_ID}.csv",
)

print("Claim checklist:")
print(claim_checklist.head(80).to_string(index=False))

# ============================================================
# 17. Manuscript text assets
# ============================================================

print("\n" + "=" * 100)
print("Step 14: Write manuscript text assets")
print("=" * 100)

def summarize_best_proxy(horizon: int, metric_delta: str, higher_is_better=True):
    sub = incremental_summary[
        (incremental_summary["horizon"] == horizon)
        & (incremental_summary["feature_set"] != "FS0_TW_only")
    ].copy()

    if sub.empty or metric_delta not in sub.columns:
        return "not available"

    sub[metric_delta] = pd.to_numeric(sub[metric_delta], errors="coerce")
    sub = sub.dropna(subset=[metric_delta])

    if sub.empty:
        return "not available"

    sub = sub.sort_values(metric_delta, ascending=not higher_is_better)
    r = sub.iloc[0]
    return f"{r['feature_set']} ({metric_delta}={r[metric_delta]:.4f})"

best_lines = []
for h in HORIZONS:
    best_lines.append(
        f"For {h}d regimes, the best macro-F1 proxy feature set was "
        f"{summarize_best_proxy(h, 'delta_test_macro_f1_vs_TW_only', True)}; "
        f"the best QWK proxy feature set was "
        f"{summarize_best_proxy(h, 'delta_test_qwk_vs_TW_only', True)}; "
        f"and the best ordinal-MAE proxy feature set was "
        f"{summarize_best_proxy(h, 'delta_test_ordinal_mae_vs_TW_only', False)}."
    )

methods_text = f"""
## Notebook 15 methods text

We evaluated whether lagged U.S. market proxy factors improve Taiwan ETF regime forecasting. The experiment used the leakage-controlled AURORA modeling dataset and constructed additional source-domain features from available market return panels. U.S. and global proxy variables were grouped into broad-market, semiconductor, risk/foreign-exchange, and regional Asia categories. To avoid time-zone leakage, every source-domain proxy series was shifted by at least one trading day before lagged and rolling features were computed. Therefore, Taiwan day t uses only U.S. and global information available no later than t-1.

Five feature sets were compared: Taiwan-only features, Taiwan plus U.S. broad-market proxies, Taiwan plus U.S. semiconductor proxies, Taiwan plus U.S. risk/foreign-exchange proxies, and Taiwan plus all U.S. proxy factors. A reference model using all existing features was also reported. For each 20-day and 60-day ordinal regime target, models were evaluated using purged and embargoed walk-forward folds. Models were selected by validation performance within each feature set, and test results were reported without using test performance for selection.
""".strip()

results_text = f"""
## Notebook 15 results text

The U.S. proxy-factor experiment compared validation-selected models across feature sets for both 20-day and 60-day ordinal regimes. The main diagnostic question was whether lagged U.S. broad-market, semiconductor, risk, or foreign-exchange proxy factors improved out-of-sample regime forecasting relative to the Taiwan-only feature set.

{chr(10).join(best_lines)}

These results should be interpreted as a forecasting extension rather than a replacement for the main AURORA allocation result. If proxy-factor improvements are weak or statistically insignificant, the paper should frame U.S. proxy factors as a tested but not necessarily decisive source of incremental information. If improvements are directionally positive and supported by paired block-bootstrap intervals, they can be described as evidence that U.S. technology and risk-market information contains useful leading state information for Taiwan semiconductor-sensitive ETF regimes.
""".strip()

limitations_text = """
## Notebook 15 limitations text

The proxy-factor experiment is subject to several limitations. First, correlation between U.S. indices and Taiwan ETFs does not imply predictive value; therefore, only lagged and walk-forward-tested proxy factors should be interpreted. Second, time-zone alignment is critical because U.S. markets close after the Taiwan market; this notebook uses a one-day lag for all U.S. proxy features to reduce look-ahead risk. Third, the Taiwan ETF strict-test period remains short, so proxy-factor improvements should be interpreted conservatively and ideally confirmed in longer samples or broader ETF universes.
""".strip()

recommended_paper_wording = """
## Recommended paper wording

As an extension, we tested whether lagged U.S. market proxy factors provide incremental information for Taiwan ETF regime forecasting. The motivation is that Taiwan semiconductor-sensitive ETFs may respond to global technology-market conditions, U.S. risk appetite, semiconductor-sector movements, exchange-rate pressure, and U.S. rate shocks. To avoid look-ahead bias from time-zone differences, all U.S. proxy variables were shifted by at least one trading day before feature construction. The results are interpreted as a source-domain proxy-factor test rather than a direct claim that U.S. indices predict Taiwan ETF prices.
""".strip()

write_markdown(MANUSCRIPT_RUN_DIR / "notebook15_methods_text.md", methods_text)
write_markdown(MANUSCRIPT_RUN_DIR / "notebook15_results_text.md", results_text)
write_markdown(MANUSCRIPT_RUN_DIR / "notebook15_limitations_text.md", limitations_text)
write_markdown(MANUSCRIPT_RUN_DIR / "notebook15_recommended_paper_wording.md", recommended_paper_wording)

write_markdown(GLOBAL_MANUSCRIPT_DIR / f"{RUN_ID}_notebook15_methods_text.md", methods_text)
write_markdown(GLOBAL_MANUSCRIPT_DIR / f"{RUN_ID}_notebook15_results_text.md", results_text)
write_markdown(GLOBAL_MANUSCRIPT_DIR / f"{RUN_ID}_notebook15_limitations_text.md", limitations_text)
write_markdown(GLOBAL_MANUSCRIPT_DIR / f"{RUN_ID}_notebook15_recommended_paper_wording.md", recommended_paper_wording)

# ============================================================
# 18. Output index
# ============================================================

print("\n" + "=" * 100)
print("Step 15: Create output index")
print("=" * 100)

output_rows = [
    {
        "artifact_type": "table",
        "name": "target_distribution_audit",
        "path": str(TABLE_RUN_DIR / "notebook15_00_target_distribution_audit.csv"),
        "description": "Target class distribution audit for 20d and 60d regimes.",
    },
    {
        "artifact_type": "table",
        "name": "available_panel_files",
        "path": str(TABLE_RUN_DIR / "notebook15_01_available_panel_files.csv"),
        "description": "Available raw panel files discovered under data/panels.",
    },
    {
        "artifact_type": "table",
        "name": "raw_proxy_group_audit",
        "path": str(TABLE_RUN_DIR / "notebook15_02_raw_proxy_group_audit.csv"),
        "description": "Raw U.S. and global proxy variables grouped by category.",
    },
    {
        "artifact_type": "table",
        "name": "lagged_proxy_feature_metadata",
        "path": str(TABLE_RUN_DIR / "notebook15_03_lagged_proxy_feature_metadata.csv"),
        "description": "Metadata for leakage-safe lagged U.S. proxy features.",
    },
    {
        "artifact_type": "table",
        "name": "feature_set_audit",
        "path": str(TABLE_RUN_DIR / "notebook15_04_feature_set_audit.csv"),
        "description": "Feature-set composition audit.",
    },
    {
        "artifact_type": "table",
        "name": "feature_list_by_set",
        "path": str(TABLE_RUN_DIR / "notebook15_05_feature_list_by_set.csv"),
        "description": "Full feature list for each feature set.",
    },
    {
        "artifact_type": "table",
        "name": "purged_walk_forward_fold_table",
        "path": str(TABLE_RUN_DIR / "notebook15_06_purged_walk_forward_fold_table.csv"),
        "description": "Purged and embargoed walk-forward fold definitions.",
    },
    {
        "artifact_type": "table",
        "name": "model_zoo",
        "path": str(TABLE_RUN_DIR / "notebook15_07_model_zoo.csv"),
        "description": "Model zoo used in Notebook 15.",
    },
    {
        "artifact_type": "table",
        "name": "fold_level_results_raw",
        "path": str(TABLE_RUN_DIR / "notebook15_08_fold_level_results_raw.csv"),
        "description": "Fold-level validation and test results.",
    },
    {
        "artifact_type": "table",
        "name": "aggregate_leaderboard_ranked",
        "path": str(TABLE_RUN_DIR / "notebook15_10_aggregate_leaderboard_ranked.csv"),
        "description": "Aggregate validation/test leaderboard.",
    },
    {
        "artifact_type": "table",
        "name": "validation_selected_feature_set_summary",
        "path": str(TABLE_RUN_DIR / "notebook15_11_validation_selected_feature_set_summary.csv"),
        "description": "Validation-selected model per feature set and horizon with test metrics.",
    },
    {
        "artifact_type": "table",
        "name": "incremental_proxy_factor_summary",
        "path": str(TABLE_RUN_DIR / "notebook15_12_incremental_proxy_factor_summary.csv"),
        "description": "Incremental proxy-factor performance relative to TW-only.",
    },
    {
        "artifact_type": "table",
        "name": "paired_block_bootstrap_feature_set_comparison",
        "path": str(TABLE_RUN_DIR / "notebook15_13_paired_block_bootstrap_feature_set_comparison.csv"),
        "description": "Paired block-bootstrap comparison against TW-only baseline.",
    },
    {
        "artifact_type": "table",
        "name": "selected_probability_export_index",
        "path": str(TABLE_RUN_DIR / "notebook15_14_selected_probability_export_index.csv"),
        "description": "Index of selected probability exports for downstream allocation experiments.",
    },
    {
        "artifact_type": "table",
        "name": "paper_selected_feature_set_forecasting_table",
        "path": str(TABLE_RUN_DIR / "notebook15_15_paper_selected_feature_set_forecasting_table.csv"),
        "description": "Paper-ready selected feature-set forecasting table.",
    },
    {
        "artifact_type": "table",
        "name": "paper_incremental_proxy_factor_table",
        "path": str(TABLE_RUN_DIR / "notebook15_16_paper_incremental_proxy_factor_table.csv"),
        "description": "Paper-ready incremental proxy-factor table.",
    },
    {
        "artifact_type": "table",
        "name": "paper_paired_bootstrap_proxy_factor_table",
        "path": str(TABLE_RUN_DIR / "notebook15_17_paper_paired_bootstrap_proxy_factor_table.csv"),
        "description": "Paper-ready paired bootstrap table.",
    },
    {
        "artifact_type": "table",
        "name": "figure_index",
        "path": str(TABLE_RUN_DIR / "notebook15_18_figure_index.csv"),
        "description": "Index of figures generated by Notebook 15.",
    },
    {
        "artifact_type": "table",
        "name": "claim_checklist",
        "path": str(TABLE_RUN_DIR / "notebook15_19_claim_checklist.csv"),
        "description": "Claim checklist for safe manuscript wording.",
    },
    {
        "artifact_type": "data",
        "name": "combined_modeling_matrix_with_lagged_us_proxies",
        "path": str(DATA_RUN_DIR / "notebook15_combined_modeling_matrix_with_lagged_us_proxies.parquet"),
        "description": "Combined modeling matrix with leakage-safe lagged U.S. proxy features.",
    },
    {
        "artifact_type": "predictions",
        "name": "all_fold_predictions",
        "path": str(PRED_RUN_DIR / "notebook15_all_fold_predictions.parquet"),
        "description": "All validation and test predictions with class probabilities.",
    },
]

for _, row in figure_index.iterrows():
    output_rows.append({
        "artifact_type": "figure",
        "name": row["figure_id"],
        "path": row["path"],
        "description": row["caption"],
    })

output_index = pd.DataFrame(output_rows)

save_table(
    output_index,
    "notebook15_output_index.csv",
    f"table_15_20_output_index_{RUN_ID}.csv",
)

print("Output index:")
print(output_index.to_string(index=False))

# ============================================================
# 19. Validation report and manifest
# ============================================================

print("\n" + "=" * 100)
print("Step 16: Save validation report and manifest")
print("=" * 100)

validation_report = {
    "project_code": COMPARISON_CODE,
    "notebook": "15_US_proxy_factor_transfer_forecasting.ipynb",
    "run_timestamp_utc": RUN_TIMESTAMP,
    "run_id": RUN_ID,
    "purpose": (
        "Leakage-controlled test of whether lagged U.S. market, semiconductor, "
        "risk, and FX proxy factors improve Taiwan ETF ordinal regime forecasting."
    ),
    "input_paths": {
        "modeling_data": str(MODELING_DATA_PATH),
        "selected_raw_return_panel": str(raw_return_path) if raw_return_path else None,
        "etf_return_panel": str(ETF_RETURN_PANEL_PATH),
    },
    "target_columns": TARGET_COLS,
    "feature_sets": {
        k: {
            "n_features": len(v),
            "n_proxy_features": int(sum(c in proxy_all_cols for c in v)),
        }
        for k, v in feature_sets.items()
    },
    "leakage_control": {
        "us_proxy_shift": "All U.S./global proxy series shifted by at least 1 day before feature use.",
        "walk_forward": "Purged and embargoed folds by horizon.",
        "embargo_days": {
            "20d": 20,
            "60d": 60,
        },
        "selection_rule": "Models selected by validation composite rank within horizon and feature set.",
    },
    "model_zoo": list(MODEL_ZOO.keys()),
    "selected_summary": selected_summary.to_dict(orient="records"),
    "incremental_summary": incremental_summary.to_dict(orient="records"),
    "claim_checklist": claim_checklist.to_dict(orient="records"),
    "output_paths": {
        "run_root": str(RUN_ROOT),
        "tables": str(TABLE_RUN_DIR),
        "data": str(DATA_RUN_DIR),
        "predictions": str(PRED_RUN_DIR),
        "probabilities": str(PROBA_RUN_DIR),
        "figures": str(PAPER_FIGURE_DIR),
        "reports": str(REPORT_RUN_DIR),
        "manuscript_assets": str(MANUSCRIPT_RUN_DIR),
    },
    "interpretation_note": (
        "Notebook 15 tests proxy-factor forecasting value only. It should not be interpreted "
        "as direct price prediction or as investment advice."
    ),
    "educational_note": (
        "This notebook is for reproducible financial machine-learning research only. "
        "It does not provide personalized financial advice or performance guarantees."
    ),
}

validation_report_path = REPORT_RUN_DIR / "NOTEBOOK15_validation_report.json"
validation_report_global_path = GLOBAL_REPORT_DIR / f"NOTEBOOK15_validation_report_{RUN_ID}.json"

write_json(validation_report_path, validation_report)
write_json(validation_report_global_path, validation_report)

manifest = make_file_manifest(RUN_ROOT)
manifest_path = REPORT_RUN_DIR / "NOTEBOOK15_file_manifest_SHA256.csv"
manifest_global_path = GLOBAL_REPORT_DIR / f"NOTEBOOK15_file_manifest_SHA256_{RUN_ID}.csv"

manifest.to_csv(manifest_path, index=False)
manifest.to_csv(manifest_global_path, index=False)

# ============================================================
# 20. Final summary
# ============================================================

print("\n" + "=" * 100)
print("NOTEBOOK 15 COMPLETE")
print("=" * 100)
print("Run ID                                      :", RUN_ID)
print("Run root                                    :", RUN_ROOT)
print("Selected raw return panel                   :", raw_return_path)
print("Modeling dataset shape                      :", model_df.shape)
print("Combined matrix shape                       :", combined_df.shape)
print("Proxy feature matrix shape                  :", proxy_features.shape)
print("Target columns                              :", TARGET_COLS)
print("Feature set audit                           :", TABLE_RUN_DIR / "notebook15_04_feature_set_audit.csv")
print("Fold table                                  :", TABLE_RUN_DIR / "notebook15_06_purged_walk_forward_fold_table.csv")
print("Aggregate leaderboard                       :", TABLE_RUN_DIR / "notebook15_10_aggregate_leaderboard_ranked.csv")
print("Selected feature-set summary                :", TABLE_RUN_DIR / "notebook15_11_validation_selected_feature_set_summary.csv")
print("Incremental proxy-factor summary            :", TABLE_RUN_DIR / "notebook15_12_incremental_proxy_factor_summary.csv")
print("Paired block bootstrap comparison           :", TABLE_RUN_DIR / "notebook15_13_paired_block_bootstrap_feature_set_comparison.csv")
print("Selected probability export index           :", TABLE_RUN_DIR / "notebook15_14_selected_probability_export_index.csv")
print("Paper selected forecasting table            :", TABLE_RUN_DIR / "notebook15_15_paper_selected_feature_set_forecasting_table.csv")
print("Paper incremental proxy-factor table        :", TABLE_RUN_DIR / "notebook15_16_paper_incremental_proxy_factor_table.csv")
print("Paper paired bootstrap table                :", TABLE_RUN_DIR / "notebook15_17_paper_paired_bootstrap_proxy_factor_table.csv")
print("Figure index                                :", TABLE_RUN_DIR / "notebook15_18_figure_index.csv")
print("Claim checklist                             :", TABLE_RUN_DIR / "notebook15_19_claim_checklist.csv")
print("Output index                                :", TABLE_RUN_DIR / "notebook15_output_index.csv")
print("Validation report                           :", validation_report_path)
print("Manifest                                    :", manifest_path)
print("=" * 100)

print("\nSelected feature-set summary:")
print(selected_summary.to_string(index=False))

print("\nIncremental proxy-factor summary:")
print(incremental_summary.to_string(index=False))

print("\nPaper-safe interpretation:")
print(
    "Notebook 15 tests whether lagged U.S. market proxy factors provide incremental "
    "forecasting value for Taiwan ETF regimes. Claims should be based on validation-selected "
    "models, test metrics, and paired block-bootstrap comparisons. Do not claim that U.S. "
    "indices directly predict Taiwan ETF prices unless the leakage-controlled evidence supports it."
)

Mounted at /content/drive
Notebook 15: U.S. Proxy-Factor Transfer Forecasting
Timestamp UTC        : 2026-06-26T01:07:49Z
Run ID               : 20260626_010749
Project root         : /content/drive/MyDrive/AURORA_TWETF
Modeling data        : /content/drive/MyDrive/AURORA_TWETF/data/modeling/AURORA_TWETF_features_with_labels.parquet
ETF return panel     : /content/drive/MyDrive/AURORA_TWETF/data/panels/AURORA_etf_return_panel.parquet
Panels directory     : /content/drive/MyDrive/AURORA_TWETF/data/panels
Run root             : /content/drive/MyDrive/AURORA_TWETF/outputs/ROMA_AURORA_TWETF/us_proxy_factor_transfer_forecasting/run_20260626_010749

Step 1: Load modeling data and identify target columns
Modeling dataset shape: (1262, 417)
Date range: 2021-01-06 00:00:00 to 2026-03-25 00:00:00
Columns preview: ['0050_ret_lag_1d', '0050_ret_lag_2d', '0050_ret_lag_3d', '0050_ret_lag_5d', '0050_ret_lag_10d', '0050_ret_lag_20d', '0050_momentum_5d_lag1', '0050_volatility_5d_lag1', '0050_ma_ratio_5